In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-credit-default-risk/sample_submission.csv
/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv
/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv
/kaggle/input/competitions/home-credit-default-risk/application_train.csv
/kaggle/input/competitions/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/competitions/home-credit-default-risk/application_test.csv
/kaggle/input/competitions/home-credit-default-risk/previous_application.csv
/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv
/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv
/kaggle/input/competitions/home-credit-default-risk/bureau.csv


In [2]:
import pandas as pd
homecredit_columns_description = pd.read_csv(
    '/kaggle/input/competitions/home-credit-default-risk/HomeCredit_columns_description.csv', 
    encoding='latin1')
pos_cash_balance = pd.read_csv(
    '/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv')
app_train = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_train.csv')
app_test  = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_test.csv')
bureau         = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/bureau.csv')
bureau_balance = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv')
credit_card    = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv')
installments   = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv')
previous_app   = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/previous_application.csv')
print("All files loaded successfully!")
print(f"Training set shape: {app_train.shape}")


All files loaded successfully!
Training set shape: (307511, 122)


In [3]:
tables={'homecredit_columns_description':homecredit_columns_description,'pos_cash_balance':pos_cash_balance,'app_train':app_train,
       'app_test':app_test,'bureau':bureau,'bureau_balance':bureau_balance,'credit_card':credit_card,'installments':installments,
       'previous_app':previous_app}

In [4]:
for name,table in tables.items():
    print(name,table.shape)

homecredit_columns_description (219, 5)
pos_cash_balance (10001358, 8)
app_train (307511, 122)
app_test (48744, 121)
bureau (1716428, 17)
bureau_balance (27299925, 3)
credit_card (3840312, 23)
installments (13605401, 8)
previous_app (1670214, 37)


In [5]:
for name,table in tables.items():
    for column in table:
        print(name,column,table[column].isna().sum(),(table[column].isna().sum()/len(table))*100.0)

homecredit_columns_description Unnamed: 0 0 0.0
homecredit_columns_description Table 0 0.0
homecredit_columns_description Row 0 0.0
homecredit_columns_description Description 0 0.0
homecredit_columns_description Special 133 60.73059360730594
pos_cash_balance SK_ID_PREV 0 0.0
pos_cash_balance SK_ID_CURR 0 0.0
pos_cash_balance MONTHS_BALANCE 0 0.0
pos_cash_balance CNT_INSTALMENT 26071 0.26067460038926715
pos_cash_balance CNT_INSTALMENT_FUTURE 26087 0.2608345786642174
pos_cash_balance NAME_CONTRACT_STATUS 0 0.0
pos_cash_balance SK_DPD 0 0.0
pos_cash_balance SK_DPD_DEF 0 0.0
app_train SK_ID_CURR 0 0.0
app_train TARGET 0 0.0
app_train NAME_CONTRACT_TYPE 0 0.0
app_train CODE_GENDER 0 0.0
app_train FLAG_OWN_CAR 0 0.0
app_train FLAG_OWN_REALTY 0 0.0
app_train CNT_CHILDREN 0 0.0
app_train AMT_INCOME_TOTAL 0 0.0
app_train AMT_CREDIT 0 0.0
app_train AMT_ANNUITY 12 0.003902299429939092
app_train AMT_GOODS_PRICE 278 0.09040327012692229
app_train NAME_TYPE_SUITE 1292 0.42014757195677555
app_train NA

In [6]:
import duckdb as db
test1=db.query("""
with compressed_pos_cash_balance as(
select distinct sk_id_curr,max(cnt_instalment) as max_instalment
from pos_cash_balance
group by sk_id_curr
),
main_table as(
select case when p.max_instalment is null then 'no pos history' when p.max_instalment > 36.0 and p.max_instalment <= 48.0 then '4th year' 
when p.max_instalment > 24.0 and p.max_instalment <= 36.0 then '3rd year' 
when p.max_instalment > 12.0 and p.max_instalment <= 24.0 then '2nd year' 
when p.max_instalment > 0.0  and p.max_instalment <= 12.0 then '1st year' 
when p.max_instalment > 48.0 then '4+ years' else 'unknown' end as cnt_instalment_year,a.amt_income_total,a.amt_credit,a.amt_annuity,a.target
from app_train a 
left join compressed_pos_cash_balance p on p.sk_id_curr=a.sk_id_curr
)
select 
cnt_instalment_year,
count(*) as total_unique_customers,
round(avg(amt_credit), 2) as avg_credit_amount,
round(avg(amt_income_total), 2) as avg_income,
round(avg(amt_annuity), 2) as avg_amount_pay,
sum(target) as total_defaulted_customers,
round(sum(target) * 100.0 / count(*), 2) as default_rate_percentage
from main_table
group by cnt_instalment_year
order by total_unique_customers desc
""").df()
test1.head(10)

,cnt_instalment_year,total_unique_customers,avg_credit_amount,avg_income,avg_amount_pay,total_defaulted_customers,default_rate_percentage
0,1st year,148313,578320.19,163432.78,27241.30,11951.0,8.06
1,2nd year,78851,590413.30,165674.05,27031.16,5975.0,7.58
2,3rd year,30858,623260.51,175742.71,27120.16,2631.0,8.53
3,4th year,18441,621777.00,179043.38,26201.63,1940.0,10.52
4,no pos history,18091,725478.43,196729.05,28767.36,1209.0,6.68
5,4+ years,12957,621795.90,179101.08,25007.53,1119.0,8.64


In [7]:
import duckdb as db
print(db.query("""
select
  count(*) as applicants,
  sum(case when days_employed = 365243 then 1 else 0 end) as sentinel_rows,
  round(avg(case when days_employed = 365243 then 1.0 else 0 end)*100, 2) as sentinel_pct,
  round(avg(case when days_employed = 365243 then target end)*100, 2) as default_pct_sentinel,
  round(avg(case when days_employed <> 365243 then target end)*100, 2) as default_pct_other
from app_train
""").df())

print(db.query("select code_gender, count(*) as n from app_train group by 1 order by n desc").df())
print(db.query("select count(*) as xna_org from app_train where organization_type = 'XNA'").df())

   applicants  sentinel_rows  sentinel_pct  default_pct_sentinel  \
0      307511        55374.0         18.01                   5.4   

   default_pct_other  
0               8.66  
  CODE_GENDER       n
0           F  202448
1           M  105059
2         XNA       4
   xna_org
0    55374


In [8]:

print(db.query("""
select count(*) as both_conditions
from app_train
where days_employed = 365243
  and organization_type = 'XNA'
""").df())


   both_conditions
0            55374


In [9]:
test2=db.query("""
with main_table as(
select sk_id_bureau,sum(case when status='C' then 1 else 0 end) as c_flag,
sum(case when status='X' then 1 else 0 end) as x_flag,
sum(case when status='0' then 1 else 0 end) as flag_0,
sum(case when status='1' then 1 else 0 end) as flag_1,
sum(case when status='2' then 1 else 0 end) as flag_2,
sum(case when status='3' then 1 else 0 end) as flag_3,
sum(case when status='4' then 1 else 0 end) as flag_4,
sum(case when status='5' then 1 else 0 end) as flag_5
from bureau_balance
group by sk_id_bureau
),
calculation_table as(
select sk_id_bureau,
x_flag,
case when (c_flag>0.0 or flag_0>0.0) and greatest(flag_1,flag_2,flag_3,flag_4,flag_5,x_flag)=0.0 then 'safe' 
when (c_flag>0) and greatest(flag_1,flag_2,flag_3)>0.0 and greatest(flag_4,flag_5)=0.0 then 'closed after delays'
when (c_flag =0.0) and greatest(flag_1,flag_2,flag_3)>0.0 and greatest(flag_4,flag_5)=0.0 then '1-3 months delay'
when (c_flag>0) and greatest(flag_4,flag_5)>0 then 'closed with heavy delays'
when (c_flag =0.0) and greatest(flag_4,flag_5)>0.0 then '3> months delay' 
else 'unclassified' end as flags
from main_table
)
select count(*)
from calculation_table
where x_flag>0 and flags='unclassified'
""").df()
test2.head(10)

,count_star()
0,398776


In [10]:
exp1=db.query("""
select  status,count(status)
from bureau_balance
group by status
""").df()
exp1.head(10)

,STATUS,count(status)
0,X,5810482
1,3,8924
2,4,5847
3,1,242347
4,5,62406
5,2,23419
6,C,13646993
7,0,7499507


In [11]:
test3=db.query("""
with main_table as(
select sk_id_bureau,sum(case when status='C' then 1 else 0 end) as c_flag,
sum(case when status='X' then 1 else 0 end) as x_flag,
sum(case when status='0' then 1 else 0 end) as flag_0,
sum(case when status='1' then 1 else 0 end) as flag_1,
sum(case when status='2' then 1 else 0 end) as flag_2,
sum(case when status='3' then 1 else 0 end) as flag_3,
sum(case when status='4' then 1 else 0 end) as flag_4,
sum(case when status='5' then 1 else 0 end) as flag_5
from bureau_balance
group by sk_id_bureau
),
calculation_table as(
select sk_id_bureau,
case when greatest(flag_1,flag_2,flag_3,flag_4,flag_5)=0 then 'none' 
when greatest(flag_1,flag_2,flag_3)>0 and greatest(flag_4,flag_5)=0 then 'mild'
when greatest(flag_4,flag_5)>0 then 'heavy' end as delay,
case when x_flag>0 then 'yes' else 'no' end as has_x,
case when c_flag>0 then 'yes' else 'no' end as has_c
from main_table
)
select delay,has_x,has_c,count(*) as count
from calculation_table
group by delay,has_x,has_c
""").df()
print(test3.shape)

(12, 4)


In [12]:
test4=db.query("""
with main_table as(
select sk_id_bureau,sum(case when status='C' then 1 else 0 end) as c_flag,
sum(case when status='X' then 1 else 0 end) as x_flag,
sum(case when status='0' then 1 else 0 end) as flag_0,
sum(case when status='1' then 1 else 0 end) as flag_1,
sum(case when status='2' then 1 else 0 end) as flag_2,
sum(case when status='3' then 1 else 0 end) as flag_3,
sum(case when status='4' then 1 else 0 end) as flag_4,
sum(case when status='5' then 1 else 0 end) as flag_5
from bureau_balance
group by sk_id_bureau
),
calculation_table as(
select sk_id_bureau,
case when greatest(flag_1,flag_2,flag_3,flag_4,flag_5)=0 then 'none' 
when greatest(flag_1,flag_2,flag_3)>0 and greatest(flag_4,flag_5)=0 then 'mild'
when greatest(flag_4,flag_5)>0 then 'heavy' end as delay,
case when x_flag>0 then 'yes' else 'no' end as has_x,
case when c_flag>0 then 'yes' else 'no' end as has_c
from main_table
),
joined_table as (
select b.sk_id_curr,c.delay,c.has_x,c.has_c
from calculation_table c 
left join bureau b on b.sk_id_bureau=c.sk_id_bureau
),
joined_table2 as(
select j.sk_id_curr,max(case when j.delay='none' then 0 when j.delay='mild' then 1 when j.delay='heavy' then 2 else 'no_history'  end) as worst_number
from joined_table j 
inner join app_train a on a.sk_id_curr=j.sk_id_curr
group by j.sk_id_curr
),
total_table as(
select count(*) as total_count
from joined_table2
)
select case when worst_number =0 then 'none' when worst_number=1 then 'mild' when worst_number=2 then 'heavy' end as delay,count(*) as count,total_count,
round((count(*)*100.0/total_count),2) as pct
from joined_table2,total_table
group by case when worst_number =0 then 'none' when worst_number=1 then 'mild' when worst_number=2 then 'heavy' end,total_count
""").df()
test4.head()

,delay,count,total_count,pct
0,none,61179,92231,66.33
1,heavy,2611,92231,2.83
2,mild,28441,92231,30.84


In [13]:
app_train.head(1)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0


In [41]:
test4b=db.query("""
with main_table as(
select sk_id_bureau,sum(case when status='C' then 1 else 0 end) as c_flag,
sum(case when status='X' then 1 else 0 end) as x_flag,
sum(case when status='0' then 1 else 0 end) as flag_0,
sum(case when status='1' then 1 else 0 end) as flag_1,
sum(case when status='2' then 1 else 0 end) as flag_2,
sum(case when status='3' then 1 else 0 end) as flag_3,
sum(case when status='4' then 1 else 0 end) as flag_4,
sum(case when status='5' then 1 else 0 end) as flag_5
from bureau_balance
group by sk_id_bureau
),
calculation_table as(
select sk_id_bureau,
case when greatest(flag_1,flag_2,flag_3,flag_4,flag_5)=0 then 'none' 
when greatest(flag_1,flag_2,flag_3)>0 and greatest(flag_4,flag_5)=0 then 'mild'
when greatest(flag_4,flag_5)>0 then 'heavy' end as delay,
case when x_flag>0 then 'yes' else 'no' end as has_x,
case when c_flag>0 then 'yes' else 'no' end as has_c
from main_table
),
joined_table as (
select b.sk_id_curr,c.delay,c.has_x,c.has_c
from calculation_table c 
left join bureau b on b.sk_id_bureau=c.sk_id_bureau
),
grouped_table as(
select sk_id_curr,max(case when delay='none' then 0 when delay='mild' then 1 when delay='heavy' then 2 else 3 end) as worst_number
from joined_table
group by sk_id_curr
),
joined_table2 as(
select a.sk_id_curr,g.worst_number,a.target
from app_train a
left join grouped_table g on a.sk_id_curr=g.sk_id_curr
),
calculated_table2 as(
select case when worst_number=0 then 'none' when worst_number=1 then 'mild' when worst_number=2 then 'heavy' else 'no_history' end as delay,
count(*) as total_count_applicants,sum(target) as defaulted_count
from joined_table2
group by case when worst_number=0 then 'none' when worst_number=1 then 'mild' when worst_number=2 then 'heavy' else 'no_history' end
)
select delay,defaulted_count,round((defaulted_count*100.0/total_count_applicants),2) as pct,
case when (defaulted_count*100.0/total_count_applicants)>8.07 then 'yes' else 'no' end as is_greater_than_default
from calculated_table2
""").df()
test4b.head()

,delay,defaulted_count,pct,is_greater_than_default
0,none,4418.0,7.22,no
1,heavy,282.0,10.80,yes
2,mild,2810.0,9.88,yes
3,no_history,17315.0,8.04,no


In [32]:
app_train.head(1)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
test5=db.query("""
with main_table as(
select code_gender,name_contract_type,count(*) as applicants,sum(target) as defaulted_count
from app_train
where code_gender!='XNA'
group by code_gender,name_contract_type
)
select code_gender,name_contract_type,defaulted_count,applicants,round((defaulted_count*100.0/applicants),2) as pct
from main_table
order by pct desc
""").df()
test5.head()